# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Choosing a simple sentence for tokenization
sample_sentence = "I am learning how BERT processes natural language."
print(f"Sample Sentence: {sample_sentence}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Sample Sentence: I am learning how BERT processes natural language.


In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


### Exercise 1 reflection
- **[CLS] and [SEP] behavior**: The `[CLS]` (Classification) token is placed at the start and serves as an aggregate representation of the entire sequence, which is crucial for tasks like sentiment analysis. The `[SEP]` (Separator) token acts as a boundary marker, telling the model where one sentence ends and another begins.
- **Attention Mask**: The attention mask is a binary tensor (1s and 0s). It prevents the model from 'looking' at the `[PAD]` tokens during the self-attention process. This ensures that the padding used to make sequences the same length doesn't distort the mathematical understanding of the actual text.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [2]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Using a clearly positive sentence for the demonstration
sentence = "I love how easy it is to use Hugging Face transformers!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence: {sentence}")
print(f"Result: {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: I love how easy it is to use Hugging Face transformers!
Result: [{'label': 'POSITIVE', 'score': 0.997262716293335}]


### Exercise 2 reflection
- **Expectation**: Yes, the predicted label (POSITIVE) matches expectations because words like 'love' and 'easy' are strong indicators of positive sentiment in the SST-2 dataset.
- **Confidence**: The model's score (typically > 0.99 for this sentence) indicates its probability of correctness. In an academic context, this score tells us how closely the input features align with the clusters the model identified as 'positive' during its fine-tuning phase.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        return self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)

    def predict(self, text: str) -> Dict[str, any]:
        self.model.eval()
        with torch.no_grad():
            inputs = self.preprocess(text)
            outputs = self.model(**inputs)
            probabilities = torch.softmax(outputs.logits, dim=1)

            score, label_idx = torch.max(probabilities, dim=1)
            label = self.model.config.id2label[label_idx.item()]

            return {"label": label, "probability": score.item()}

In [4]:
# Instantiate and test the analyzer
analyzer = BERTSentimentAnalyzer()
samples = [
    "This BERT tutorial is absolutely fantastic and easy to follow!",
    "I am quite disappointed with the complexity of this implementation."
]

for text in samples:
    result = analyzer.predict(text)
    print(f"Text: {text}")
    print(f"Result: {result}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This BERT tutorial is absolutely fantastic and easy to follow!
Result: {'label': 'POSITIVE', 'probability': 0.9998654127120972}

Text: I am quite disappointed with the complexity of this implementation.
Result: {'label': 'NEGATIVE', 'probability': 0.9997946619987488}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [5]:
from transformers import pipeline

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.ner_pipeline = pipeline("ner", model=model_name, aggregation_strategy="simple")

    def recognize(self, text: str):
        return self.ner_pipeline(text)

In [6]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin in California."
entities = ner.recognize(sample_text)

for ent in entities:
    print(f"Entity: {ent['word']}, Label: {ent['entity_group']}, Confidence: {ent['score']:.4f}")

# Reflection: How subwords are handled
# In this implementation, the aggregation_strategy='simple' handles subwords (like '##ing')
# by averaging the scores of the constituent tokens and merging them back into a single word span.

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Entity: Google, Label: ORG, Confidence: 0.9985
Entity: Larry Page, Label: PER, Confidence: 0.9998
Entity: Sergey Brin, Label: PER, Confidence: 0.9990
Entity: California, Label: LOC, Confidence: 0.9997


## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only (Bi-directional) | Decoder-only (Uni-directional/Auto-regressive) |
| Primary purpose | Understanding & Contextual Representation | Text Generation & Completion |
| Typical use cases | Classification, NER, Q&A | Chatbots, Creative Writing, Summarization |
| Strengths | Deeply understands context from both sides | Excellent at maintaining flow and generating text |
| Weaknesses | Cannot generate long sequences of text | Traditionally 'blind' to future tokens in a sequence |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **BERT for Encoding**: BERT is used as a 'Bi-Encoder' in RAG. It transforms raw text into dense vector embeddings. Because BERT is bi-directional, these embeddings capture nuanced semantic meaning, allowing the system to understand that 'Who is the CEO?' and 'Who leads the company?' are semantically similar even if they use different words.

2. **Vector Databases**: These BERT embeddings are stored in a specialized database (like Pinecone or FAISS). When a user asks a question, BERT encodes the query, and the system performs a mathematical 'cosine similarity' search to find the document vectors closest to the query vector.

3. **Handing to the Generator**: Once BERT identifies the top-K most relevant document snippets, these are concatenated with the original user prompt and sent to a Decoder model like GPT. This provides the 'context' the generator needs to avoid hallucinations.

4. **Application Example**: A Technical Support Bot for a software company. BERT retrieves specific documentation pages based on the user's error message, and GPT synthesizes those technical steps into a conversational, helpful guide for the customer.